In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy import constants as const

from mcfacts.inputs.settings_manager import AGNDisk, SettingsManager
from mcfacts.utilities import unit_conversion


In [ ]:
a = [1000]
b = [1100]
np.isclose(a, b, atol=100)

In [ ]:
sirko_agn_disk = AGNDisk(SettingsManager(
    {
        "disk_model_name": "sirko_goodman"
    }
))

thompson_agn_disk = AGNDisk(SettingsManager(
    {
        "disk_model_name": "thompson_etal"
    }
))

In [ ]:
def stahler_drag(mass_1, mass_2, bin_sep, orb_a, disk_sound_speed, disk_density, timestep_duration_yr, smbh_mass, r_g_in_meters):
    q = np.minimum(mass_1 / mass_2, mass_2 / mass_1)

    total_mass = ((mass_1 + mass_2) * const.M_sun).si

    scaling_constant = (15 / (35 * np.pi))
    ratio_component = (((1 + q) ** 2) / q)
    gas_component = (((disk_sound_speed(orb_a) * u.meter/u.second) ** 5) / (disk_density(orb_a) * (u.kg / u.m ** 3)))
    mass_component = 1 / ((const.G ** 3) * (total_mass ** 2))

    sep_unit = unit_conversion.si_from_r_g(smbh_mass, bin_sep, r_g_defined=r_g_in_meters)

    coalescence_time = (scaling_constant * ratio_component * gas_component * mass_component) * sep_unit

    timestep_units = (timestep_duration_yr * u.year).si

    return bin_sep * (1 - (timestep_units / coalescence_time))

In [ ]:
disk_rg = np.linspace(10, 50000, 1000)

s_disk = sirko_agn_disk.disk_density(disk_rg)
t_disk = thompson_agn_disk.disk_density(disk_rg)

plt.plot(disk_rg, s_disk, label="sirko")
plt.plot(disk_rg, t_disk, label="thompson")
plt.loglog()

plt.legend()

In [ ]:
disk_rg = np.linspace(10, 50000, 1000)

s_disk = sirko_agn_disk.disk_sound_speed(disk_rg)
t_disk = thompson_agn_disk.disk_sound_speed(disk_rg)

plt.plot(disk_rg, s_disk, label="sirko")
plt.plot(disk_rg, t_disk, label="thompson")
plt.loglog()

plt.legend()

In [ ]:
mass_1 = np.array([10.0])
mass_2 = np.array([10.0])
bin_sep = np.array([100])
orb_a = np.array([10000.0])

stahler_drag(mass_1, mass_2, bin_sep, orb_a, thompson_agn_disk.disk_sound_speed, thompson_agn_disk.disk_density, thompson_agn_disk.settings.active_timestep_duration_yr, thompson_agn_disk.settings.smbh_mass, thompson_agn_disk.settings.r_g_in_meters)